<a href="https://colab.research.google.com/github/darwish012/ML-52-5930/blob/main/lab7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install "tensorflow==2.17.*" "numpy>=1.23,<2.0" tensorflow-model-optimization


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 601.4/601.4 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.5/242.5 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 31.7 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.6
    Uninstalling protobuf-5.29.6:
      Successfully uninstalled protobuf-5.29.6
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: tensorboard
    Found existing installation: tensorboard 2.19.0
    Uninstalling tensorboard-2.19.0:
      Succ

In [1]:
pip install "ml_dtypes>=0.5"


In [2]:
import tensorflow as tf  # Imports the core TensorFlow library used for building and training the neural networks.
import tensorflow_model_optimization as tfmot  # Imports the optimization toolkit containing pruning, clustering, and QAT tools.
from tensorflow import keras  # Imports the Keras API, a high-level interface for building models easily in TensorFlow.
import numpy as np  # Imports NumPy for numerical operations, specifically used later for the SVD math.
import os  # Imports the OS module to interact with the operating system, used here to check file sizes on disk.
import tempfile  # Imports the tempfile module to create temporary directories for saving models safely to measure their size.
from tensorflow.lite.python.util import convert_bytes_to_c_source  # Imports a utility to convert the raw TFLite binary into C++ code for microcontrollers.
# ==========================================
# HELPER — measure model size on disk
# ==========================================
def get_model_size_kb(model, label):  # Defines a helper function that takes a model and a text label as input.
    """Save model to a temp file and return its size in KB."""  # Docstring explaining the function's purpose.
    with tempfile.TemporaryDirectory() as tmp:  # Creates a temporary folder on the operating system that automatically deletes itself when done.
        path = os.path.join(tmp, "model.keras")  # Creates a full file path string inside the temporary folder named 'model.keras'.
        model.save(path)  # Saves the Keras model to that temporary path on the hard drive.
        size_kb = os.path.getsize(path) / 1024  # Gets the exact file size in bytes from the OS, then divides by 1024 to convert to Kilobytes (KB).
    print(f"  📦 [{label}] Model size: {size_kb:.1f} KB  ({size_kb/1024:.2f} MB)")  # Prints the formatted size in both KB and MB for the students to see.
    return size_kb  # Returns the calculated size in KB so it can be stored in our dictionary.

def print_section(title):  # Defines a simple helper function to print formatted section headers in the console.
    print("\n" + "="*60)  # Prints a newline followed by a top border of 60 equals signs.
    print(f"  {title}")  # Prints the section title indented by two spaces.
    print("="*60)  # Prints a bottom border of 60 equals signs.
def recompile(model):  # FIX: helper to recompile after any strip_* call.
    # strip_pruning() and strip_clustering() both return a NEW model object
    # that has lost its compile config. Call this immediately after any strip.
    model.compile(
        optimizer='adam',
        loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=['accuracy']
    )
    return model
sizes = {}   # Initializes an empty dictionary to store the file sizes at each stage so we can build the final summary table.


In [3]:
# ==========================================
# DATA
# ==========================================
print_section("LOADING DATA")  # Calls the helper function to print the data loading header.
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()  # Downloads and loads the MNIST dataset (handwritten digits), splitting it into training and testing sets.
x_train = (x_train / 255.0).astype("float32")[..., None]  # Normalizes training pixel values to be between 0.0 and 1.0, casts to float32, and adds a channel dimension for the Conv2D layer.
x_test  = (x_test  / 255.0).astype("float32")[..., None]  # Normalizes testing pixel values exactly like the training set, and adds the channel dimension.
print(f"  Train samples : {len(x_train):,}")  # Prints the total number of training images.
print(f"  Test  samples : {len(x_test):,}")  # Prints the total number of testing images.
print(f"  Input shape   : {x_train.shape[1:]}")  # Prints the shape of a single image (28, 28, 1).



  LOADING DATA
11490434/11490434 [==============================] - 2s 0us/step
  Train samples : 60,000
  Test  samples : 10,000
  Input shape   : (28, 28, 1)


In [ ]:

# ==========================================
# LARGE TEACHER MODEL (~100 MB)
# We use many large Dense layers so students can see dramatic compression.
# ==========================================
print_section("BUILDING & TRAINING TEACHER MODEL  (~100 MB)")  # Prints the header for the teacher model phase.
print("  (Large model intentionally — shows why compression matters)")  # Prints context for the students explaining why the model is so massive.

def build_teacher():  # Defines a function to construct the massive teacher model architecture.
    return keras.Sequential([  # Returns a Keras Sequential model, where layers are stacked one after another linearly.
        keras.layers.Input((28, 28, 1)),  # Defines the input shape matching the MNIST images (28x28 pixels, 1 grayscale channel).
        keras.layers.Conv2D(32, 3, activation='relu'),  # Adds a 2D Convolutional layer with 32 filters of size 3x3, using ReLU activation.
        keras.layers.MaxPooling2D(),  # Adds a Max Pooling layer to downsample the spatial dimensions (reduces size by half).
        keras.layers.Flatten(),  # Flattens the 2D feature maps into a 1D vector to feed into the Dense layers.
        # Large fully-connected layers to inflate size to ~100 MB
        keras.layers.Dense(4096, activation='relu'),  # Adds a massive Dense (fully connected) layer with 4,096 neurons.
        keras.layers.Dense(4096, activation='relu'),  # Adds a second massive Dense layer with 4,096 neurons.
        keras.layers.Dense(4096, activation='relu'),  # Adds a third massive Dense layer with 4,096 neurons to heavily inflate parameter count.
        keras.layers.Dense(10)  # The final output layer with 10 neurons, corresponding to the 10 digit classes (0-9). No softmax here so it outputs raw logits.
    ], name="teacher")  # Gives the model the explicit name "teacher".

teacher = build_teacher()  # Calls the function and instantiates the teacher model.
teacher.compile(  # Configures the model for training.
    optimizer='adam',  # Uses the Adam optimizer, an efficient gradient descent algorithm.
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),  # Uses crossentropy loss for classification; from_logits=True because we didn't use softmax in the last layer.
    metrics=['accuracy']  # Tells the model to track classification accuracy during training.
)
print(f"\n  Teacher parameters: {teacher.count_params():,}")  # Prints the total number of trainable parameters (weights and biases) in the massive teacher.
teacher.fit(x_train, y_train, epochs=2, batch_size=256, verbose=1)  # Trains the teacher model on the training data for 2 epochs, updating weights in batches of 256.
_, teacher_acc = teacher.evaluate(x_test, y_test, verbose=0)  # Evaluates the trained teacher on the unseen test data. We ignore the loss (_) and keep the accuracy.
print(f"  Teacher test accuracy: {teacher_acc*100:.2f}%")  # Prints the baseline accuracy of the large model for the students to compare against later.
sizes['0. Teacher (initial)'] = get_model_size_kb(teacher, "Teacher — before compression")  # Measures the huge file size and saves it to the sizes dictionary.
teacher.trainable = False  # Freezes the teacher's weights so they don't accidentally get updated during the distillation process later.



  BUILDING & TRAINING TEACHER MODEL  (~100 MB)
  (Large model intentionally — shows why compression matters)

  Teacher parameters: 55,759,178
Epoch 1/2
235/235 [==============================] - 558s 2s/step - loss: 0.1882 - accuracy: 0.9426
Epoch 2/2
178/235 [=====================>........] - ETA: 2:15 - loss: 0.0461 - accuracy: 0.9859

In [ ]:
teacher.save_weights("teacher_weights.h5")
#teacher = build_teacher()
#teacher.load_weights("teacher_weights.h5")

In [ ]:

# ==========================================
# SMALL STUDENT MODEL
# ==========================================
print_section("BUILDING STUDENT MODEL  (small — to be trained via distillation)")  # Prints header for the student model phase.

def build_student():  # Defines a function to construct the tiny student model.
    return keras.Sequential([  # Returns a Sequential model for the student.
        keras.layers.Input((28, 28, 1)),  # Same input shape as the teacher (28x28 grayscale image).
        keras.layers.Conv2D(8, 3, activation='relu'),  # Uses a much smaller Conv2D layer with only 8 filters (teacher had 32).
        keras.layers.MaxPooling2D(),  # Standard max pooling to reduce spatial dimensions.
        keras.layers.Flatten(),  # Flattens the output into a 1D vector.
        keras.layers.Dense(32, activation='relu'),  # Uses a tiny Dense layer with only 32 neurons (teacher had 4096!).
        keras.layers.Dense(10)  # Output layer with 10 neurons for the 10 classes, outputting raw logits.
    ], name="student")  # Names this model "student".

student = build_student()  # Instantiates the student model.
print(f"  Student parameters: {student.count_params():,}")  # Prints the parameter count of the student (very small).
print(f"  That is {teacher.count_params()//student.count_params():,}x fewer parameters than the teacher!")  # Calculates and prints the massive ratio difference in size for the students.


In [ ]:

# ==========================================
# DISTILLATION
# The student learns from the teacher's soft probability outputs,
# not just hard class labels — it inherits the teacher's "knowledge".
# ==========================================
print_section("STEP 1 — KNOWLEDGE DISTILLATION")  # Prints the header for the knowledge distillation step.
print("  Student learns from teacher's soft outputs (knowledge transfer).")  # Explains the concept of soft targets.
print("  Alpha=0.3  ->  30% hard labels, 70% teacher guidance.")  # Explains how alpha weights the two different loss functions.
print("  Temperature T=4  ->  softens probabilities for richer signal.\n")  # Explains how temperature makes the teacher's output distribution less extreme.

class Distiller(keras.Model):  # Creates a custom Keras Model subclass specifically to override the training logic for distillation.
    def __init__(self, student, teacher):  # Constructor method that takes both the student and teacher models.
        super().__init__()  # Calls the parent class (keras.Model) constructor to initialize internal Keras mechanics.
        self.student = student  # Stores the student model as a class attribute.
        self.teacher = teacher  # Stores the teacher model as a class attribute.

    def compile(self, optimizer, loss_fn, distill_fn, alpha=0.3, T=4):  # Custom compile method to accept extra distillation hyperparameters.
        super().compile()  # Calls parent compile method.
        self.optimizer  = keras.optimizers.get(optimizer)  # Retrieves and stores the optimizer object (like Adam).
        self.loss_fn    = loss_fn  # Stores the standard loss function (for hard ground-truth labels).
        self.distill_fn = distill_fn  # Stores the distillation loss function (usually KL Divergence for comparing probability distributions).
        self.alpha      = alpha  # Stores the alpha weight (controls balance between hard and soft loss).
        self.T          = T  # Stores the Temperature factor used to soften the logits.

    def train_step(self, data):  # Overrides the core training loop step; this defines exactly what happens in one batch update.
        x, y = data  # Unpacks the batch data into 'x' (images) and 'y' (true labels).
        teacher_pred = self.teacher(x, training=False)  # Passes images through the teacher to get its predictions. training=False ensures dropout/batchnorm act in inference mode.
        with tf.GradientTape() as tape:  # Opens a GradientTape context to record operations for automatic differentiation (calculating gradients).
            student_pred = self.student(x, training=True)  # Passes images through the student model to get its predictions.
            loss_hard = self.loss_fn(y, student_pred)  # Calculates standard loss between student predictions and true labels.
            loss_soft = self.distill_fn(  # Calculates the distillation loss...
                tf.nn.softmax(teacher_pred / self.T),  # ...by applying softmax to the teacher's logits divided by Temperature (soft targets)...
                tf.nn.softmax(student_pred / self.T)  # ...and comparing it to the student's logits also softened by Temperature.
            ) * (self.T ** 2)  # Multiplies soft loss by T-squared to scale gradients correctly relative to the hard loss.
            loss = self.alpha * loss_hard + (1 - self.alpha) * loss_soft  # Combines both losses using the alpha weighting.
        grads = tape.gradient(loss, self.student.trainable_variables)  # Calculates the gradients of the combined loss with respect to the student's weights.
        self.optimizer.apply_gradients(zip(grads, self.student.trainable_variables))  # Uses the optimizer to update the student's weights based on the gradients.
        return {"loss": loss}  # Returns a dictionary containing the loss value so Keras can track it in the progress bar.

distiller = Distiller(student, teacher)  # Instantiates our custom Distiller class with our specific student and teacher.
distiller.compile(  # Compiles the distiller.
    optimizer='adam',  # Uses Adam optimizer.
    loss_fn=keras.losses.SparseCategoricalCrossentropy(from_logits=True),  # Standard classification loss for the hard labels.
    distill_fn=keras.losses.KLDivergence()  # Uses Kullback-Leibler Divergence to measure how similar the student's probability distribution is to the teacher's.
)
distiller.fit(x_train, y_train, epochs=2, batch_size=256, verbose=1)  # Kicks off the custom distillation training loop for 2 epochs.

model = student  # Now that distillation is done, we extract the trained student and assign it to the variable 'model' for the rest of the pipeline.

# Student was trained inside Distiller which has its own compile —
# we need to compile the raw student directly before calling evaluate().
model.compile(
    optimizer='adam',
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

_, acc = model.evaluate(x_test, y_test, verbose=0)
print(f"\n  Student test accuracy after distillation: {acc*100:.2f}%")
sizes['1. After distillation'] = get_model_size_kb(model, "After distillation")
_, acc = model.evaluate(x_test, y_test, verbose=0)  # Evaluates the purely distilled student on the test set.
print(f"\n  Student test accuracy after distillation: {acc*100:.2f}%")  # Prints the accuracy. It should be surprisingly good despite the small size!
sizes['1. After distillation'] = get_model_size_kb(model, "After distillation")  # Measures and records the model size (which dropped dramatically just by changing architectures).


In [ ]:
# ==========================================
# STEP 2 — PRUNING
# Sets unimportant weights to zero (sparsity).
# 80% final sparsity = 80% of weights become 0.
# ==========================================
print_section("STEP 2 — WEIGHT PRUNING")
print("  Gradually zeroes out unimportant weights during training.")
print("  Schedule: 30% -> 80% sparsity over 1000 steps.")
print("  Result: most weights become zero — compressible storage.\n")

prune_sched = tfmot.sparsity.keras.PolynomialDecay(
    initial_sparsity=0.3,  # Start: zero out the 30% smallest weights immediately.
    final_sparsity=0.8,    # End: 80% of all weights will be zero by step 1000.
    begin_step=0,
    end_step=1000
)
model = tfmot.sparsity.keras.prune_low_magnitude(model, prune_sched)  # Wraps model with pruning logic.
model.compile(
    optimizer='adam',
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)
model.fit(
    x_train, y_train,
    epochs=3, batch_size=256, verbose=1,
    callbacks=[tfmot.sparsity.keras.UpdatePruningStep()]  # CRITICAL: updates sparsity masks every batch.
)
model = tfmot.sparsity.keras.strip_pruning(model)  # Removes pruning wrappers, leaving sparse weight matrices.

# FIX: strip_pruning() returns a new model object that has lost its compile config.
model = recompile(model)
_, acc = model.evaluate(x_test, y_test, verbose=0)
print(f"\n  Test accuracy after pruning: {acc*100:.2f}%")
sizes['2. After pruning'] = get_model_size_kb(model, "After pruning")


In [ ]:

# ==========================================
# STEP 3 — CLUSTERING
# Groups weights into N clusters — each weight stores only a
# cluster index (4 bits for 16 clusters) instead of a full float32.
# ==========================================
print_section("STEP 3 — WEIGHT CLUSTERING")
print("  Groups all weights into 16 clusters.")
print("  Each weight replaced by its nearest centroid value.")
print("  Storage: 4-bit cluster index instead of 32-bit float = 8x saving.\n")

model = tfmot.clustering.keras.cluster_weights(
    model,
    number_of_clusters=16,  # Only 16 unique weight values allowed per layer.
    cluster_centroids_init=tfmot.clustering.keras.CentroidInitialization.LINEAR  # Spread initial centroids evenly across weight range.
)
model.compile(
    optimizer='adam',
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)
model.fit(x_train, y_train, epochs=3, batch_size=256, verbose=1)  # Fine-tunes centroids to recover accuracy.
model = tfmot.clustering.keras.strip_clustering(model)  # Removes clustering wrappers, weights now only have 16 unique values.

# FIX: strip_clustering() returns a new model object that has lost its compile config.
model = recompile(model)
_, acc = model.evaluate(x_test, y_test, verbose=0)
print(f"\n  Test accuracy after clustering: {acc*100:.2f}%")
sizes['3. After clustering'] = get_model_size_kb(model, "After clustering")


In [ ]:

# ==========================================
# QAT — Quantization Aware Training
# Simulates INT8 arithmetic during training so the model learns
# to be robust to the precision loss before actual quantization.
# ==========================================
print_section("STEP 4 — QUANTIZATION AWARE TRAINING (QAT)")  # Prints QAT header.
print("  Simulates INT8 (8-bit integer) arithmetic during training.")  # Explains what QAT actually does.
print("  Float32 = 32 bits per weight  ->  INT8 = 8 bits = 4x smaller.")  # Explains the math of the size reduction.
print("  QAT trains the model to tolerate this precision loss gracefully.\n")  # Explains why we train instead of just converting directly.

model = tfmot.quantization.keras.quantize_model(model)  # Wraps the model to insert "FakeQuant" nodes that simulate 8-bit rounding errors during the forward pass.
model.compile(  # Re-compiles the QAT model.
    optimizer='adam',
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)
model.fit(x_train, y_train, epochs=3, batch_size=256, verbose=1)  # Trains the model, forcing it to adjust its weights to survive the simulated INT8 precision limits.
_, acc = model.evaluate(x_test, y_test, verbose=0)  # Evaluates QAT accuracy.
print(f"\n  Test accuracy after QAT: {acc*100:.2f}%")  # Prints QAT accuracy.
sizes['4. After QAT'] = get_model_size_kb(model, "After QAT")  # Measures and records size (note: file is still technically float32 here, true INT8 happens at conversion).



In [ ]:

# ==========================================
# SVD — Low-Rank Approximation on Dense Head
# Decomposes the final weight matrix W into U*S*Vt and keeps
# only the top-k singular values — discards least important directions.
# ==========================================
print_section("STEP 5 — SVD LOW-RANK APPROXIMATION (Dense head)")  # Prints SVD header.
print("  Factorises the final Dense weight matrix W (32x10)")  # Explains which layer we are attacking.
print("  into U * S * Vt using Singular Value Decomposition.")  # Explains the mathematical tool.
print("  Keeping only the top k=4 singular values (rank-4 approximation).")  # Explains the compression factor 'k'.
print("  Original params: 32x10 = 320  ->  SVD params: (32x4)+(4x4)+(4x10) = 184\n")  # Proves the math of parameter reduction to the students.

all_weights = model.layers[-1].get_weights()  # Grabs the actual trained weights and biases from the very last Dense layer of our model.
W_raw, b    = all_weights[0], all_weights[1]  # Splits the list into the Weight matrix (W_raw) and the bias vector (b).

if W_raw.ndim == 1:  # Safety check in case the weight matrix flattened unexpectedly.
    n_out = b.shape[0]  # Infers output size from bias.
    n_in  = W_raw.shape[0] // n_out  # Infers input size mathematically.
    W = W_raw.reshape(n_in, n_out)  # Reshapes it back to a proper 2D matrix.
else:  # If it is already a 2D matrix (which it should be)...
    W = W_raw  # ...just assign it to W.

U, S, Vt = np.linalg.svd(W, full_matrices=False)  # Performs Singular Value Decomposition using NumPy. W becomes U (left singular vectors), S (singular values), and Vt (right singular vectors).
k = min(4, S.shape[0])  # Sets our target rank k to 4, making sure we don't accidentally ask for more rank than exists.

print(f"  Singular values (importance scores): {np.round(S, 3)}")  # Prints the singular values so students can see how they decay (first is large, last is small).
print(f"  Keeping top {k} — discarding {len(S)-k} least important directions.\n")  # Explains exactly how much information is being thrown away.

U_k = U[:, :k].astype("float32")  # Slices the U matrix to keep only the first 'k' columns, and casts to float32 for TensorFlow compatibility.
S_k = np.diag(S[:k]).astype("float32")  # Takes the top 'k' singular values, turns them into a diagonal matrix, and casts to float32.
V_k = Vt[:k, :].astype("float32")  # Slices the Vt matrix to keep only the first 'k' rows, and casts to float32.

def svd_layer(x):  # Defines a custom function representing our new, factorized layer.
    x = tf.matmul(x, U_k)  # Multiplies incoming data by the truncated U matrix.
    x = tf.matmul(x, S_k)  # Multiplies that result by the diagonal S matrix.
    x = tf.matmul(x, V_k)  # Multiplies that result by the truncated Vt matrix.
    return x + b  # Adds the original bias vector back to the final result.

inputs  = model.input  # Grabs the input tensor from our existing model.
x       = model.layers[-2].output  # Grabs the output tensor from the second-to-last layer (right before the Dense layer we replaced).
outputs = svd_layer(x)  # Passes that output through our custom SVD math operations to get the final predictions.

model   = keras.Model(inputs, outputs)  # Stitches everything together into a brand new Keras model with the SVD head attached.
model = recompile(model)

_, acc  = model.evaluate(x_test, y_test, verbose=0)  # Evaluates the model to see if throwing away SVD components hurt accuracy.
print(f"  Test accuracy after SVD: {acc*100:.2f}%")  # Prints SVD accuracy.
sizes['5. After SVD'] = get_model_size_kb(model, "After SVD")  # Measures and records the model size.


In [ ]:

# ==========================================
# INT8 TFLITE CONVERSION
# ==========================================
print_section("STEP 6 — INT8 TFLite CONVERSION")  # Prints TFLite conversion header.
print("  Converts the Keras model to TensorFlow Lite format.")  # Explains the tool being used.
print("  All weights quantized to INT8 — ready for microcontrollers.\n")  # Explains the final data type.

def rep_data():  # Defines a Python generator function to provide a "representative dataset".
    for i in range(1000):  # Loops through the first 1000 images of the training set.
        yield [x_train[i:i+1]]  # Yields them one by one. The converter uses this data to calibrate the min/max ranges for the INT8 activation quantization.

converter = tf.lite.TFLiteConverter.from_keras_model(model)  # Creates a TFLiteConverter object from our final Keras model.
converter.optimizations             = [tf.lite.Optimize.DEFAULT]  # Enables default optimizations (which includes weight quantization).
converter.representative_dataset    = rep_data  # Passes our generator function to calibrate activations to INT8 based on real data ranges.
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]  # Strictly forces the converter to ONLY use INT8 operations (fails if an op requires float).
converter.inference_input_type      = tf.int8  # Forces the model to expect INT8 inputs instead of floats (crucial for microcontrollers).
converter.inference_output_type     = tf.int8  # Forces the model to output INT8 values.

tflite_model = converter.convert()  # Executes the conversion, turning the Python graph into a raw flatbuffer binary.

with open("model.tflite", "wb") as f:  # Opens a file named 'model.tflite' in 'write-binary' mode.
    f.write(tflite_model)  # Saves the flatbuffer binary to the hard drive.

tflite_kb = os.path.getsize("model.tflite") / 1024  # Calculates the true final size of the optimized binary.
sizes['6. Final INT8 TFLite'] = tflite_kb  # Records the final size.
print(f"  INT8 TFLite model size: {tflite_kb:.2f} KB  ({tflite_kb/1024:.3f} MB)")  # Prints the incredibly small final size.

# ==========================================
# TFLITE MICRO — C source export
# ==========================================
print_section("STEP 7 — TFLite MICRO C SOURCE EXPORT")  # Prints the C export header.
print("  Converts .tflite binary to a C array for embedding")  # Explains why we do this step.
print("  directly into microcontroller firmware (Arduino, STM32, etc.).\n")  # Connects the software to IoT hardware concepts.

cc, hh = convert_bytes_to_c_source(tflite_model, "mnist_model")  # Uses the utility tool to translate the binary bits into a readable C-style byte array.

with open("model.cc", "w") as f:  # Opens a C++ source file.
    f.write(cc)  # Writes the massive byte array into the .cc file.
with open("model.h", "w") as f:  # Opens a C++ header file.
    f.write(hh)  # Writes the necessary declarations into the .h file.

flash_kb = (os.path.getsize("model.cc") + os.path.getsize("model.h")) / 1024  # Calculates the space this will take up in microcontroller flash memory.
print(f"  Flash size (C source, model.cc + model.h): {flash_kb:.2f} KB")  # Prints the flash memory footprint.


In [ ]:

# ==========================================
# FINAL SUMMARY TABLE
# ==========================================
print_section("FINAL COMPRESSION SUMMARY")  # Prints the summary header.
initial_kb = sizes['0. Teacher (initial)']  # Retrieves the massive teacher's size.
final_kb   = sizes['6. Final INT8 TFLite']  # Retrieves the tiny TFLite model's size.

print(f"\n  {'Stage':<30} {'Size (KB)':>10}  {'Size (MB)':>10}  {'vs. Teacher':>14}")  # Prints the column headers for the table.
print(f"  {'-'*68}")  # Prints a separator line.
for label, kb in sizes.items():  # Loops through the dictionary containing all recorded sizes.
    mb = kb / 1024  # Converts KB to MB for display.
    if kb < initial_kb:  # If the current stage is smaller than the teacher...
        ratio = f"{initial_kb/kb:.0f}x smaller"  # ...calculate the compression ratio.
    else:  # Otherwise (which only applies to the teacher itself)...
        ratio = "(baseline)"  # ...label it as the baseline.
    print(f"  {label:<30} {kb:>9.1f}  {mb:>9.3f}  {ratio:>14}")  # Prints the formatted row for the table.

print(f"\n  {'='*68}")  # Prints a bottom border for the table.
print(f"  Initial teacher model : {initial_kb:>9.1f} KB  =  {initial_kb/1024:.1f} MB")  # Highlights the starting point.
print(f"  Final TFLite model    : {final_kb:>9.2f} KB  =  {final_kb/1024:.3f} MB")  # Highlights the ending point.
print(f"  Total compression     : {initial_kb/final_kb:.0f}x smaller")  # Displays the total multi-stage compression multiplier.
print(f"  Size reduction        : {(1 - final_kb/initial_kb)*100:.1f}% of original size removed")  # Displays the percentage of data removed.
print(f"  {'='*68}\n")  # Prints a final closing border.
print("  Files saved:")  # Indicates the outputs generated by the script.
print("    model.tflite  — deploy to Android / embedded Linux")  # Explains where the flatbuffer goes.
print("    model.cc      — embed in MCU firmware (Arduino, STM32)")  # Explains where the C source file goes.
print("    model.h       — header for MCU firmware\n")  # Explains where the C header file goes.